In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.feature_selection import RFE, chi2, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

# ===========================================================================
# 1. PATHS & DATA
# ===========================================================================
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

X_train_raw = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test_raw  = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")
y_train     = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test      = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

print(f"Original features: {X_train_raw.shape[1]}")

# ===========================================================================
# 2. ROTTERDAM CRITERIA FEATURE ENGINEERING
# (Rotterdam criteria: PCOS diagnosed via any 2 of 3 — oligo/anovulation,
#  hyperandrogenism, polycystic ovaries on ultrasound. These engineered
#  features encode that clinical logic numerically.)
# ===========================================================================
def apply_rotterdam_features(df):
    df_feat = df.copy()

    # Follicle-related (polycystic ovary marker)
    if {'Follicle No. (L)', 'Follicle No. (R)'}.issubset(df_feat.columns):
        df_feat['Total_Follicles'] = df_feat['Follicle No. (L)'] + df_feat['Follicle No. (R)']
        df_feat['Max_Follicle_Count'] = np.maximum(df_feat['Follicle No. (L)'], df_feat['Follicle No. (R)'])
        df_feat['Follicle_Asymmetry'] = np.abs(df_feat['Follicle No. (L)'] - df_feat['Follicle No. (R)'])

    if {'Avg. F size left (mm)', 'Avg. F size right (mm)'}.issubset(df_feat.columns):
        df_feat['Avg_Follicle_Size'] = (df_feat['Avg. F size left (mm)'] + df_feat['Avg. F size right (mm)']) / 2

    # Hormonal ratio (hyperandrogenism / anovulation marker)
    if {'LH(mIU/mL)', 'FSH(mIU/mL)'}.issubset(df_feat.columns):
        df_feat['LH_FSH_Ratio'] = df_feat['LH(mIU/mL)'] / (df_feat['FSH(mIU/mL)'] + 1e-5)

    if {'AMH(ng/mL)', 'BMI'}.issubset(df_feat.columns):
        df_feat['AMH_BMI_Product'] = df_feat['AMH(ng/mL)'] * df_feat['BMI']

    if {'AMH(ng/mL)', 'Weight (kg)'}.issubset(df_feat.columns):
        df_feat['AMH_Weight_Interaction'] = df_feat['AMH(ng/mL)'] * df_feat['Weight (kg)']

    # Metabolic / weight-related markers (commonly elevated in PCOS)
    if {'Weight (kg)', 'Height(Cm)'}.issubset(df_feat.columns):
        height_m = df_feat['Height(Cm)'] / 100
        df_feat['Recomputed_BMI'] = df_feat['Weight (kg)'] / (height_m ** 2 + 1e-5)

    if {'Waist(inch)', 'Hip(inch)'}.issubset(df_feat.columns):
        df_feat['Waist_Hip_Ratio'] = df_feat['Waist(inch)'] / (df_feat['Hip(inch)'] + 1e-5)

    return df_feat

X_train_eng = apply_rotterdam_features(X_train_raw)
X_test_eng  = apply_rotterdam_features(X_test_raw)

print(f"After Rotterdam feature engineering: {X_train_eng.shape[1]} features")

# ===========================================================================
# 3. METHOD 1 — CORRELATION WITH TARGET
# ===========================================================================
corr_with_target = X_train_eng.apply(lambda col: col.corr(y_train)).abs().sort_values(ascending=False)
n_keep = min(15, X_train_eng.shape[1])
correlation_top = set(corr_with_target.head(n_keep).index)

# ===========================================================================
# 4. METHOD 2 — CHI-SQUARE (requires non-negative values, so scale first)
# ===========================================================================
scaler_chi = MinMaxScaler()
X_train_scaled_chi = pd.DataFrame(
    scaler_chi.fit_transform(X_train_eng), columns=X_train_eng.columns, index=X_train_eng.index
)
chi_selector = SelectKBest(score_func=chi2, k=n_keep)
chi_selector.fit(X_train_scaled_chi, y_train)
chi_square_top = set(X_train_eng.columns[chi_selector.get_support()])

# ===========================================================================
# 6. CONSENSUS VOTING
# Features selected by BOTH Correlation and Chi-Square methods are retained
# for model development.
# ===========================================================================
all_features = list(X_train_eng.columns)
vote_counts = {
    feat: sum([feat in correlation_top, feat in chi_square_top])
    for feat in all_features
}

vote_df = pd.DataFrame({
    "Feature": list(vote_counts.keys()),
    "Votes": list(vote_counts.values()),
    "In_Correlation_Top": [f in correlation_top for f in vote_counts],
    "In_ChiSquare_Top": [f in chi_square_top for f in vote_counts],
}).sort_values("Votes", ascending=False)

print("\nFeature voting summary:")
print(vote_df)

# Keep only the features selected by BOTH feature selection methods.
consensus_features = vote_df[vote_df["Votes"] >= 2]["Feature"].tolist()

# Safety fallback to ensure a sufficient number of informative features
# if the consensus set becomes too small.
if len(consensus_features) < 5:
    print("\nWarning: very few features reached 2+ votes — relaxing to top-15 by vote then correlation")
    vote_df["corr_strength"] = vote_df["Feature"].map(corr_with_target)
    consensus_features = vote_df.sort_values(["Votes", "corr_strength"], ascending=[False, False]).head(15)["Feature"].tolist()
elif len(consensus_features) > 15:
    vote_df["corr_strength"] = vote_df["Feature"].map(corr_with_target)
    consensus_features = vote_df[vote_df["Votes"] >= 2].sort_values(["Votes", "corr_strength"], ascending=[False, False]).head(15)["Feature"].tolist()

print(f"\nFinal consensus feature count: {len(consensus_features)}")
print("Selected features:", consensus_features)

# ===========================================================================
# 7. BUILD FINAL SELECTED DATASETS
# ===========================================================================
X_train_selected_imp = X_train_eng[consensus_features].copy()
X_test_selected_imp = X_test_eng[consensus_features].copy()

# ===========================================================================
# 8. SAVE FILES
# ===========================================================================
X_train_selected_imp.to_csv(PROCESSED_DIR / "X_train_selected_imp.csv", index=False)
X_test_selected_imp.to_csv(PROCESSED_DIR / "X_test_selected_imp.csv", index=False)
vote_df.to_csv(PROCESSED_DIR / "feature_voting_summary.csv", index=False)

print(f"\nSaved: {PROCESSED_DIR / 'X_train_selected_imp.csv'}  shape={X_train_selected_imp.shape}")
print(f"Saved: {PROCESSED_DIR / 'X_test_selected_imp.csv'}  shape={X_test_selected_imp.shape}")
print(f"Saved: {PROCESSED_DIR / 'feature_voting_summary.csv'}")

Original features: 41
After Rotterdam feature engineering: 47 features

Feature voting summary:
                   Feature  Votes  In_Correlation_Top  In_ChiSquare_Top
8               Cycle(R/I)      2                True              True
9       Cycle length(days)      2                True              True
36        Follicle No. (L)      2                True              True
28        hair growth(Y/N)      2                True              True
41         Total_Follicles      2                True              True
42      Max_Follicle_Count      2                True              True
43      Follicle_Asymmetry      2                True              True
31            Pimples(Y/N)      2                True              True
32         Fast food (Y/N)      2                True              True
37        Follicle No. (R)      2                True              True
22              AMH(ng/mL)      2                True              True
27        Weight gain(Y/N)      2       

In [ ]:
# ===========================================================================
# 9. TRIM REDUNDANT FOLLICLE FEATURES (NEW CELL)
# ===========================================================================

# Define the exact features we want to drop to stop repeating data
features_to_drop = ['Total_Follicles', 'Max_Follicle_Count']

# Drop columns from the training and testing selected datasets
X_train_selected_imp = X_train_selected_imp.drop(columns=features_to_drop, errors='ignore')
X_test_selected_imp = X_test_selected_imp.drop(columns=features_to_drop, errors='ignore')

# Update the consensus features list tracking variable
consensus_features = [f for f in consensus_features if f not in features_to_drop]

# Overwrite the saved files with the new lean 11-feature datasets
X_train_selected_imp.to_csv(PROCESSED_DIR / "X_train_selected_imp.csv", index=False)
X_test_selected_imp.to_csv(PROCESSED_DIR / "X_test_selected_imp.csv", index=False)

# Print the final shape and updated feature list to confirm success
print(f"Updated Train Shape: {X_train_selected_imp.shape}")
print(f"Updated Test Shape:  {X_test_selected_imp.shape}")
print("\nFinal clean features:", consensus_features)


Updated Train Shape: (378, 11)
Updated Test Shape:  (163, 11)

Final clean features: ['Cycle(R/I)', 'Cycle length(days)', 'Follicle No. (L)', 'hair growth(Y/N)', 'Follicle_Asymmetry', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Follicle No. (R)', 'AMH(ng/mL)', 'Weight gain(Y/N)', 'Skin darkening (Y/N)']


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report,
    RocCurveDisplay, log_loss
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CM_DIR = OUTPUTS_DIR / "baseline_14features" / "confusion_matrices"
ROC_DIR = OUTPUTS_DIR / "baseline_14features" / "roc_curves"
TABLE_DIR = OUTPUTS_DIR / "comparison_tables"
REPORT_DIR = OUTPUTS_DIR / "baseline_14features" / "classification_reports"
for p in [MODELS_DIR, CM_DIR, ROC_DIR, TABLE_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ===========================================================================
# DATA — 14-FEATURE (SELECTED) VERSION, NO TUNING
# ===========================================================================
X_train = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("+", "plus")
        .replace("[", "")
        .replace("]", "")
        .replace(":", "")
    )

def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return model.predict(X)

def optimize_threshold(y_true, scores, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_threshold = 0.5
    best_value = -1
    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        value = f1_score(y_true, preds, zero_division=0) if metric == "f1" else fbeta_score(y_true, preds, beta=2, zero_division=0)
        if value > best_value:
            best_value = value
            best_threshold = threshold
    return float(best_threshold), float(best_value)

def metric_block(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    out = {
        "Accuracy": accuracy_score(y_true, preds),
        "Error Rate": 1 - accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "F2": fbeta_score(y_true, preds, beta=2, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores),
    }
    try:
        clipped = np.clip(scores, 1e-6, 1 - 1e-6)
        out["Log Loss"] = log_loss(y_true, clipped)
    except Exception:
        out["Log Loss"] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    out.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return out, preds

def plot_confusion(y_true, preds, title, path):
    cm = confusion_matrix(y_true, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_roc(y_true, scores, title, path):
    plt.figure(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_true, scores)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def evaluate_model(model, model_name, phase, threshold_source="train"):
    model.fit(X_train, y_train)
    train_scores = predict_scores(model, X_train)
    test_scores = predict_scores(model, X_test)
    threshold, threshold_metric = optimize_threshold(y_train, train_scores, metric="f1")
    train_metrics, train_preds = metric_block(y_train, train_scores, threshold)
    test_metrics, test_preds = metric_block(y_test, test_scores, threshold)
    cv_acc = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1)
    cv_f1 = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="f1", n_jobs=1)
    try:
        cv_auc = cross_val_score(clone(model), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    except Exception:
        cv_auc = np.array([np.nan])

    name = safe_name(f"{phase}_{model_name}")
    plot_confusion(y_test, test_preds, f"{phase} {model_name} Confusion Matrix (14 Features)", CM_DIR / f"{name}_cm.png")
    plot_roc(y_test, test_scores, f"{phase} {model_name} ROC Curve (14 Features)", ROC_DIR / f"{name}_roc.png")
    (REPORT_DIR / f"{name}_classification_report.txt").write_text(classification_report(y_test, test_preds, target_names=["No PCOS", "PCOS"], zero_division=0))
    joblib.dump(model, MODELS_DIR / f"{name}.joblib")
    (MODELS_DIR / f"{name}_features.json").write_text(
    json.dumps(list(X_train.columns))
)

    row = {
        "Model": model_name,
        "Phase": phase,
        "Threshold": threshold,
        "CV Accuracy Mean": np.nanmean(cv_acc),
        "CV Accuracy Std": np.nanstd(cv_acc),
        "CV F1 Mean": np.nanmean(cv_f1),
        "CV F1 Std": np.nanstd(cv_f1),
        "CV ROC-AUC Mean": np.nanmean(cv_auc),
        "CV ROC-AUC Std": np.nanstd(cv_auc),
    }
    row.update({f"Train {k}": v for k, v in train_metrics.items()})
    row.update({f"Test {k}": v for k, v in test_metrics.items()})
    return row, model, test_scores

# ===========================================================================
# BASELINE MODEL DEFINITIONS — SAME DEFAULTS AS YOUR 41-FEATURE BASELINE
# ===========================================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
    "Gaussian NB": GaussianNB(),
    "Bagging": BaggingClassifier(estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42), n_estimators=150, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=150, learning_rate=0.5, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=21),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),
    "Perceptron": CalibratedClassifierCV(Perceptron(class_weight="balanced", random_state=42), cv=5),
}
if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42
    )
else:
    models["XGBoost"] = GradientBoostingClassifier(random_state=43)

models["Stacking ML"] = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    cv=5,
    n_jobs=1,
)

print(f"Total baseline models: {len(models)}")
print(list(models))

# ===========================================================================
# TRAINING LOOP
# ===========================================================================
results = []
fitted_models = {}
roc_scores = {}

for name, model in models.items():
    print(f"Training baseline model (11 features): {name}")
    row, fitted, test_scores = evaluate_model(model, name, "Baseline_Selected")
    results.append(row)
    fitted_models[name] = fitted
    roc_scores[name] = test_scores

baseline_results_selected = pd.DataFrame(results)
baseline_results_selected["Rank"] = baseline_results_selected["Test F1"].rank(ascending=False, method="min").astype(int)
baseline_results_selected = baseline_results_selected.sort_values(["Rank", "Test ROC-AUC"], ascending=[True, False])
baseline_results_selected.to_csv(TABLE_DIR / "baseline_results_14features.csv", index=False)
display(baseline_results_selected[["Rank", "Model", "Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC", "CV Accuracy Mean"]])

Train: (378, 11) | Test: (163, 11)
Total baseline models: 14
['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'Gaussian NB', 'Bagging', 'AdaBoost', 'Gradient Boosting', 'KNN', 'LDA', 'QDA', 'Perceptron', 'XGBoost', 'Stacking ML']
Training baseline model (11 features): Logistic Regression
Training baseline model (11 features): Decision Tree
Training baseline model (11 features): Random Forest
Training baseline model (11 features): SVM
Training baseline model (11 features): Gaussian NB
Training baseline model (11 features): Bagging
Training baseline model (11 features): AdaBoost
Training baseline model (11 features): Gradient Boosting
Training baseline model (11 features): KNN
Training baseline model (11 features): LDA
Training baseline model (11 features): QDA
Training baseline model (11 features): Perceptron
Training baseline model (11 features): XGBoost
Training baseline model (11 features): Stacking ML


,Rank,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,CV Accuracy Mean
0,1,Logistic Regression,0.907975,0.839286,0.886792,0.862385,0.949228,0.896728
6,2,AdaBoost,0.907975,0.931818,0.773585,0.845361,0.947513,0.899502
4,3,Gaussian NB,0.901840,0.877551,0.811321,0.843137,0.951286,0.896728
11,4,Perceptron,0.895706,0.875000,0.792453,0.831683,0.928816,0.907397
9,5,LDA,0.889571,0.857143,0.792453,0.823529,0.946141,0.912731
12,6,XGBoost,0.883436,0.814815,0.830189,0.822430,0.946827,0.888976
10,6,QDA,0.883436,0.814815,0.830189,0.822430,0.942710,0.891607
13,8,Stacking ML,0.883436,0.840000,0.792453,0.815534,0.947170,0.907326
3,9,SVM,0.877301,0.800000,0.830189,0.814815,0.944082,0.896799
8,10,KNN,0.889571,0.926829,0.716981,0.808511,0.935420,0.891607


<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

...

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

<Figure size 600x500 with 0 Axes>

# Imports & Paths

In [9]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, log_loss, RocCurveDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"

# All outputs live under outputs/with_FE_baseline/<model_name>/
BASE_OUT      = PROJECT_ROOT / "outputs" / "with_FE_baseline"
TABLE_DIR     = PROJECT_ROOT / "outputs" / "comparison_tables"

for p in [MODELS_DIR, TABLE_DIR, BASE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Output root:", BASE_OUT)

Output root: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline


# Load Data

In [10]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
X_test  = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Target distribution (train):\n{y_train.value_counts()}")

Train : (378, 11)
Test  : (163, 11)
Target distribution (train):
PCOS
0    254
1    124
Name: count, dtype: int64


#  Helper Functions

In [11]:
def safe_name(name):
    return (name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "")
            .replace("+", "plus").replace(":", ""))


def model_dir(model_name):
    """Return (and create) outputs/with_FE_baseline/<model_name>/"""
    d = BASE_OUT / safe_name(model_name)
    d.mkdir(parents=True, exist_ok=True)
    return d


def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X).astype(float)


def optimize_threshold(y_true, scores, metric="f1"):
    best_t, best_v = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (scores >= t).astype(int)
        v = (f1_score(y_true, preds, zero_division=0) if metric == "f1"
             else fbeta_score(y_true, preds, beta=2, zero_division=0))
        if v > best_v:
            best_v, best_t = v, t
    return float(best_t), float(best_v)


def metric_block(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    out = {
        "Accuracy"  : accuracy_score(y_true, preds),
        "Error Rate": 1 - accuracy_score(y_true, preds),
        "Precision" : precision_score(y_true, preds, zero_division=0),
        "Recall"    : recall_score(y_true, preds, zero_division=0),
        "F1"        : f1_score(y_true, preds, zero_division=0),
        "F2"        : fbeta_score(y_true, preds, beta=2, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_true, scores),
        "PR-AUC"    : average_precision_score(y_true, scores),
    }
    try:
        out["Log Loss"] = log_loss(y_true, np.clip(scores, 1e-6, 1 - 1e-6))
    except Exception:
        out["Log Loss"] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    out.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return out, preds


# ── Plot: Confusion Matrix ─────────────────────────────────────────────────
def plot_confusion(y_true, preds, model_name, out_dir):
    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax)
    ax.set_title(f"with FE Baseline — {model_name}\nConfusion Matrix", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    path = out_dir / "confusion_matrix.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def plot_roc(y_true, scores, model_name, out_dir):
    auc = roc_auc_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # 1. Pass an empty or generic string to name to prevent standard concatenation conflicts
    disp = RocCurveDisplay.from_predictions(y_true, scores, name="", ax=ax)
    
    # 2. Directly overwrite the line label with your precise 4-decimal string
    disp.line_.set_label(f"Classifier (AUC = {auc:.4f})")
    
    # 3. Re-draw the legend to apply your custom label formatting
    ax.legend(loc="lower right")
    
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(f"with FE Baseline — {model_name}\nROC Curve", fontsize=11, fontweight="bold")
    plt.tight_layout()
    path = out_dir / "roc_curve.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


# ── Plot: Stratified 10-Fold CV ────────────────────────────────────────────
def plot_stratified_cv(model, model_name, out_dir):
    fold_acc, fold_f1, fold_auc = [], [], []
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    for fold_i, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
        m = clone(model)
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        sc = predict_scores(m, X_train.iloc[va])
        th, _ = optimize_threshold(y_train.iloc[tr],
                                   predict_scores(m, X_train.iloc[tr]))
        preds = (sc >= th).astype(int)
        fold_acc.append(accuracy_score(y_train.iloc[va], preds))
        fold_f1.append(f1_score(y_train.iloc[va], preds, zero_division=0))
        try:
            fold_auc.append(roc_auc_score(y_train.iloc[va], sc))
        except Exception:
            fold_auc.append(np.nan)

    folds = np.arange(1, 11)
    fig, ax = plt.subplots(figsize=(12, 5))
    for vals, label, color, marker in [
        (fold_acc, f"Accuracy  (mean={np.nanmean(fold_acc):.4f})", "#1f77b4", "o"),
        (fold_f1,  f"F1        (mean={np.nanmean(fold_f1):.4f})",  "#ff7f0e", "s"),
        (fold_auc, f"ROC-AUC   (mean={np.nanmean(fold_auc):.4f})", "#2ca02c", "^"),
    ]:
        ax.plot(folds, vals, marker=marker, label=label, linewidth=1.8)
        for x, y in zip(folds, vals):
            ax.annotate(f"{y:.4f}", (x, y),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=7.5, color="dimgray")

    ax.set_xticks(folds)
    ax.set_xticklabels([f"Fold {i}" for i in folds])
    ax.set_ylabel("Score")
    ax.set_ylim(max(0, min(fold_acc + fold_f1 + fold_auc) - 0.08), 1.08)
    ax.set_title(
        f"with FE Baseline — {model_name}\n"
        f"10-Fold Stratified CV ({X_train.shape[1]} Features)",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    path = out_dir / "stratified_cv.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

print("Helper functions defined.")

Helper functions defined.


# Model Definitions

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42),

    "SVM": SVC(
        kernel="rbf", probability=True, class_weight="balanced", random_state=42),

    "Gaussian NB": GaussianNB(),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced", random_state=42),
        n_estimators=150, random_state=42),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=150, learning_rate=0.5, random_state=42),

    "Gradient Boosting": GradientBoostingClassifier(random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=21),

    "LDA": LinearDiscriminantAnalysis(),

    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),

    "Perceptron": CalibratedClassifierCV(
        Perceptron(class_weight="balanced", random_state=42), cv=5),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42)
else:
    models["XGBoost"] = GradientBoostingClassifier(random_state=43)

models["Stacking ML"] = StackingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
        ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ("gb",  GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42),
    cv=5, n_jobs=1,
)

print(f"Total models: {len(models)}")
print(list(models.keys()))

Total models: 14
['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'Gaussian NB', 'Bagging', 'AdaBoost', 'Gradient Boosting', 'KNN', 'LDA', 'QDA', 'Perceptron', 'XGBoost', 'Stacking ML']


# Training Loop

In [13]:
results = []

for name, model in models.items():
    print(f"\n▶ Training : {name}")
    t_train_start = time.time()
    model.fit(X_train, y_train)
    train_time_sec = round(time.time() - t_train_start, 4)
    out_dir = model_dir(name)

    # ── Fit ───────────────────────────────────────────────────────────────
    model.fit(X_train, y_train)
    train_scores = predict_scores(model, X_train)
    test_scores  = predict_scores(model, X_test)

    threshold, _ = optimize_threshold(y_train, train_scores, metric="f1")

    train_metrics, train_preds = metric_block(y_train, train_scores, threshold)
    test_metrics,  test_preds  = metric_block(y_test,  test_scores,  threshold)

    # ── CV scores ─────────────────────────────────────────────────────────
    cv_acc = cross_val_score(clone(model), X_train, y_train,
                             cv=cv, scoring="accuracy", n_jobs=1)
    cv_f1  = cross_val_score(clone(model), X_train, y_train,
                             cv=cv, scoring="f1",       n_jobs=1)
    try:
        cv_auc = cross_val_score(clone(model), X_train, y_train,
                                 cv=cv, scoring="roc_auc", n_jobs=1)
    except Exception:
        cv_auc = np.array([np.nan])

    # ── Plots ─────────────────────────────────────────────────────────────
    plot_confusion(y_test, test_preds,  name, out_dir)
    plot_roc(y_test,       test_scores, name, out_dir)
    plot_stratified_cv(model,           name, out_dir)

    # ── Save model ────────────────────────────────────────────────────────
    joblib.dump(model, MODELS_DIR / f"with_fe_baseline_{safe_name(name)}.joblib")
    (MODELS_DIR / f"with_fe_baseline_{safe_name(name)}_features.json").write_text(
        json.dumps(list(X_train.columns)))

    # ── Results row ───────────────────────────────────────────────────────
    row = {
        "Model"              : name,
        "Threshold"          : threshold,
        "CV Accuracy Mean"   : np.nanmean(cv_acc),
        "CV Accuracy Std"    : np.nanstd(cv_acc),
        "CV F1 Mean"         : np.nanmean(cv_f1),
        "CV F1 Std"          : np.nanstd(cv_f1),
        "CV ROC-AUC Mean"    : np.nanmean(cv_auc),
        "CV ROC-AUC Std"     : np.nanstd(cv_auc),
        "train_time_sec"     : train_time_sec,
    }
    row.update({f"Train {k}": v for k, v in train_metrics.items()})
    row.update({f"Test {k}" : v for k, v in test_metrics.items()})
    results.append(row)
    print(f"  ✓ Done | Acc={test_metrics['Accuracy']:.4f} "
          f"F1={test_metrics['F1']:.4f} AUC={test_metrics['ROC-AUC']:.4f}")


▶ Training : Logistic Regression
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\logistic_regression\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\logistic_regression\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\logistic_regression\stratified_cv.png
  ✓ Done | Acc=0.9080 F1=0.8624 AUC=0.9492

▶ Training : Decision Tree
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\decision_tree\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\decision_tree\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_baseline\decision_tr

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Define explicit base directory dynamically
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
BASE_OUT.mkdir(parents=True, exist_ok=True)

# 2. Build the results DataFrame directly from your active notebook run
df_results = pd.DataFrame(results)
df_results["Rank"] = df_results["Test F1"].rank(ascending=False, method="min").astype(int)
df_results = df_results.sort_values(["Rank", "Test ROC-AUC"], ascending=[True, False])

# 3. Display the tuned performance table
display(df_results[[
    "Rank", "Model",
    "Test Accuracy", "Test Precision", "Test Recall",
    "Test F1", "Test ROC-AUC",
    "CV F1 Mean", "CV ROC-AUC Mean"
]])

# 4. Save your brand new tuned outputs cleanly to disk
df_results.to_csv(BASE_OUT / "with_FE_Baseline_results.csv", index=False)

print("\nAll done. Outputs saved under:", BASE_OUT)


,Rank,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,CV F1 Mean,CV ROC-AUC Mean
0,1,Logistic Regression,0.907975,0.839286,0.886792,0.862385,0.949228,0.848869,0.952474
6,2,AdaBoost,0.907975,0.931818,0.773585,0.845361,0.947513,0.840720,0.941615
4,3,Gaussian NB,0.901840,0.877551,0.811321,0.843137,0.951286,0.850229,0.960487
11,4,Perceptron,0.895706,0.875000,0.792453,0.831683,0.928816,0.849073,0.947346
9,5,LDA,0.889571,0.857143,0.792453,0.823529,0.946141,0.863904,0.956256
12,6,XGBoost,0.883436,0.814815,0.830189,0.822430,0.946827,0.825724,0.954218
10,6,QDA,0.883436,0.814815,0.830189,0.822430,0.942710,0.837091,0.947808
13,8,Stacking ML,0.883436,0.840000,0.792453,0.815534,0.947170,0.861548,0.959833
3,9,SVM,0.877301,0.800000,0.830189,0.814815,0.944082,0.842305,0.954833
8,10,KNN,0.889571,0.926829,0.716981,0.808511,0.935420,0.817555,0.951519



All done. Outputs saved under: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs
